In [1]:
import pandas as pd
import numpy as np
import re
import math
import os

In [2]:
area = {'reading_area': {'center': (918.2, 584.4), 'width': 1186.3, 'height': 889.1}, 
       'video_list': {'center': (162, 330), 'width': 304, 'height': 302.5},
       'ir-reading_area1': {'center': (165, 714), 'width': 305.8, 'height': 413.4},
       'ir-reading_area2': {'center': (757, 68), 'width': 1504.5, 'height': 123.2},
       'ir-reading_area3': {'center': (1716, 92), 'width': 384.1, 'height': 168},
       'ir-reading_area4': {'center': (1718.3, 925.5), 'width': 377, 'height': 161.6},
       'note_area': {'center': (1724, 623), 'width': 353.9, 'height': 394.4},
       'planning_area': {'center': (1714, 300), 'width': 372.4, 'height': 219.5},
       'timer1': {'center': (97, 989), 'width': 166.8, 'height': 73.3},
       'timer2': {'center': (1851, 1051), 'width': 96.7, 'height': 49.9}
       }
for a in area.keys():
    left = area[a]['center'][0] - 0.5 * area[a]['width']
    right = area[a]['center'][0] + 0.5 * area[a]['width']
    bottom = area[a]['center'][1] - 0.5 * area[a]['height']
    top = area[a]['center'][1] + 0.5 * area[a]['height']
    area[a]['range'] = [left, right, bottom, top]    

In [3]:
def preprocess(filename, log_name):
    df_eye = pd.read_csv(filename, skiprows=5)
    df_eye = df_eye[['Timestamp', 'Datetime', 'Gaze X', 'Gaze Y']]
    df_eye = df_eye.dropna(how='any')
    df_eye['Timestamp'] = df_eye['Timestamp']/1000
    df_eye['Datetime'] = pd.to_datetime(df_eye['Datetime'], errors='coerce')
    df_eye['Datetime'] = df_eye['Datetime'].dt.strftime('%Y-%m-%d %H:%M:%S')
    gaze_area = []
    for index, row in df_eye.iterrows():
        x = row['Gaze X']
        y = row['Gaze Y']
        area_name = 'gap'
        for a in area.keys():
            area_range = area[a]['range']
            if area_range[0] < x < area_range[1] and area_range[2] < y < area_range[3]:
                area_name = a
                break
        gaze_area.append(area_name)
    df_eye['area'] = gaze_area
    date = ''
    df = {'time': [], 'area': []}
    last_area = ''
    time_area = []
    for index, row in df_eye.iterrows():
        if not date:
            date = row['Datetime']
            last_area = row['area']
            time_area.append(row['area'])
        elif date != row['Datetime']:
            df['time'].append(date)
            df['area'].append(time_area)
            time_area = [row['area']]
            last_area = row['area']
            date = row['Datetime']
        else:
            if last_area != row['area']:
                last_area = row['area']
                time_area.append(last_area)
    df['time'].append(date)
    df['area'].append(time_area)
    df = pd.DataFrame(df)
    df_action = pd.read_excel(log_name)
    df_action = (df_action.groupby('time', as_index=False).agg({'action': ', '.join}))
    df_final = pd.merge(df_action, df, how='outer', on='time')
    cur_stat = 'not started'
    status = []
    for index, row in df_final.iterrows():
        action = row['action']
        if type(action) != str and math.isnan(action):
            pass
        elif action == 'start':
            cur_stat = 'start'
        elif action == 'play_video':
            cur_stat = 'video_playing'
        elif action == 'pause_video':
            cur_stat = 'video_not_playing'
        elif action == 'Video 1' or action == 'Video 2' or action == 'Instructions' or action == 'read instruction':
            cur_stat = 'video_not_playing'
        elif re.match(r'\d_*', action):
            cur_stat = 'video_not_playing'
        status.append(cur_stat)
    df_final['status'] = status
    df_final['time'] = pd.to_datetime(df_final['time'])
    full_range = pd.date_range(start=df_final['time'].min(), end=df_final['time'].max(), freq='s')
    df_tempt = pd.DataFrame({'time': full_range})
    df_final = pd.merge(df_tempt, df_final, on='time', how='left')
    df_final['status'] = df_final['status'].fillna('missing')
    return df_final

In [4]:
# Rule 1: notes + reading_area/note_area + video_not_playing = note_editing
# Rule 2: notes + reading_area/note_area + video_playing = note_editing + video_play
# Rule 6: quiz + any area + any status = quiz_answer
# Rule 7: quiz multiple times before switch page = quiz_retry
# Rule 8: quiz reading is not possible to deduced from log because it is the same behaviour as watching video
# Rule 9: video_pause follow by video play immediately = video skip or rewind
# Rule 10: look at reading area for more than 3 seconds = content reading
# Rule 11: look at notes area for more than 3 seconds = notes_reading
# Rule 12: look at reading area for more than 3 seconds + video_not_playing = video_pause_review

In [4]:
class Rule:
    def __init__(self, action = '', area = [], status = '', label = []):
        self.action = action
        self.area = area
        self.status = status
        self.label = label

    def check(self, row):
        action = row['action'].split(', ')
        if row['status'] == 'not started':
            return []
        if self.action != 'any' and self.action not in action:
            return []
        if len(action) > 1:
            return []
        if self.area != 'any' and not set(row['area']).intersection(set(self.area)):
            return []
        if self.status != 'any' and row['status'] != self.status:
            return []
        return self.label

class DurationRule(Rule):
    def __init__(self, action = '', area = [], status = '', label = [], threshold = 0, end = [], last_action = ''):
        super().__init__(action, area, status, label)
        self.duration = 0
        self.threshold = threshold
        self.end = end
        self.last_action = last_action

    def check(self, row):
        basic_check = super().check(row)
        action = row['action'].split(', ')
        if action:
            self.last_action = action[-1]
        if basic_check:
            self.duration += 1
        else:
            self.duration = 0
        if self.last_action in self.end:
            self.duration = 0
        if self.duration == self.threshold:
            return [self.label * (self.threshold - 1)] + self.label
        if self.duration > self.threshold:
            return self.label
        return []

class PauseReviewRule(Rule):
    def __init__(self, action = '', area = [], status = '', label = [], threshold = 0, signal = []):
        super().__init__(action, area, status, label)
        self.duration = 0
        self.threshold = threshold
        self.change_page = False
        self.signal = signal

    def check(self, row):
        basic_check = super().check(row)
        action = row['action'].split(', ')
        for a in action:
            if a in self.signal:
                self.change_page = True
                break
        if row['status'] == 'video_playing':
            self.change_page = False
        if not self.change_page and basic_check:
            self.duration += 1
        else:
            self.duration = 0
        if self.duration == self.threshold:
            return [self.label * (self.threshold - 1)] + self.label
        if self.duration > self.threshold:
            return self.label
        return []

class RetryRule(Rule):
    def __init__(self, action = '', area = [], status = '', label = []):
        super().__init__(action, area, status, label)
        self.viewed_page = {}
        self.current_page = ''
        self.left_page = {}

    def check(self, row):
        action = row['action'].split(', ')
        for a in action:
            if a in switch_page and self.current_page and a != self.current_page:
                self.left_page[self.current_page] = 1
            if a in switch_page and a not in self.viewed_page.keys():
                self.viewed_page[a] = 1
                self.current_page = a
            elif a in self.viewed_page and a in self.left_page.keys():
                self.viewed_page[a] += 1
                self.current_page = a
            if self.current_page and self.viewed_page[self.current_page] > 1 and a == self.action and len(action) == 1:
                return self.label
        return []
        
class ComboRule(Rule):
    def __init__(self, action = '', area = [], status = '', label = [], combo = []):
        super().__init__(action, area, status, label)
        self.combo = combo
        self.tempt = combo[:]

    def check(self, row):
        action = row['action'].split(', ')
        if len(self.tempt) != 0:        
            if self.tempt[0] in action:
                self.tempt.pop(0)
            else:
                self.tempt = self.combo[:]
        if len(self.tempt) == 0:  
            self.tempt = self.combo[:]
            return self.label
        return []
        
class NavigationRule(Rule):
    def __init__(self, action = '', area = [], status = '', label = [], area_count = 1):
        super().__init__(action, area, status, label)
        self.area_count = area_count

    def check(self, row):
        basic_check = super().check(row)
        if not basic_check:
            return []
        if self.area == "any" and len(set(row['area'])) >= self.area_count:
            return self.label
        if self.area != "any" and len(set(row['area']).intersection(set(self.area))) >= self.area_count:
            return self.label
        return []

class ProgressCheckRule(Rule):
    def __init__(self, action = '', area = [], status = '', label = [], signal_area = []):
        super().__init__(action, area, status, label)
        self.last_area = ''
        self.current_area = []
        self.signal_area = signal_area

    def check(self, row):
        self.current_area = row['area']
        if set(row['area']).intersection(set(self.area)) and set(self.last_area).intersection(set(self.signal_area)):
            return self.label
        self.last_area = row['area']
        return []
        

In [5]:
note = 'notes'
plan = 'plan'
plan_area = 'plan_area'
note_area = 'note_area'
timer1 = 'timer1'
timer2 = 'timer2'
time_checking = 'time_checking'
reading_area = 'reading_area'
status_on = 'video_playing'
status_off = 'video_not_playing'
plan_editing = 'plan_editing'
note_editing = 'note_editing'
note_reading = 'note_reading'
plan_reading = 'plan_reading'
content_reading = 'content_reading'
quiz = 'quiz'
anything = 'any'
quiz_answer = 'quiz_attempt'
quiz_retry = 'quiz_retry'
video_pause_review = 'video_pause_review'
play_video = 'play_video'
pause_video = 'pause_video'
video_skip = 'video skip or rewind'
switch_page = ['Instructions', 'Video 1', 'Video 2','03_q1a', '03_q1b', '03_q1c', '03_q2c', '03_q2b', '03_q2c', '03_q3a', '03_q3b', '03_q3c', '03_q4a', '03_q4b', '03_q4c', '03_q5a', '03_q5b', '03_q5c', '04_q1a', '04_q1b', '04_q2a', '04_q2b', '04_q3a', '04_q3b', '04_q4a', '04_q4b', '04_q5a', '04_q5b']
irrelevant_area = ["ir-reading_area1", "ir-reading_area2", "ir-reading_area3"]
content_navigation = 'content_navigation'
ir_content_navigation = 'irrelevant_content_navigation'
ir_reading = 'irrelevant_reading'
play_video = 'play_video'
progress_check = 'progress_review'
video_list = 'video_list'
def make_rules():
    # note editing
    rule1 = Rule(action = note, area = anything, status = status_off, label = [note_editing])
    # note editing + content reading
    rule2 = Rule(action = note, area = anything, status = status_on, label = [note_editing, content_reading])
    # quiz answer
    rule3 = Rule(action = quiz, area = anything, status = anything, label = [quiz_answer])
    # content reading
    rule4 = DurationRule(action = anything, area = [reading_area], status = anything, label = [content_reading], threshold = 3)
    # note reading
    rule5 = DurationRule(action = '', area = [note_area], status = anything, label = [note_reading], threshold = 3)
    # pause review
    rule6 = PauseReviewRule(action = '', area = [reading_area], status = status_off, label = [video_pause_review], threshold = 2, signal = switch_page)
    # quiz retry
    rule7 = RetryRule(action = quiz, area = anything, status = anything, label = [quiz_retry])
    # video skip
    rule8 = ComboRule(action = anything, area = anything, status = anything, label = [video_skip], combo = [pause_video, play_video])
    # plan editing
    rule9 = Rule(action = plan, area = anything, status = anything, label = [plan_editing])
    # plan reading
    rule10 = DurationRule(action = '', area = [plan_area], status = anything, label = [plan_reading], threshold = 3)
    # time checking
    rule11 = Rule(action = anything, area = [timer1, timer2], status = anything, label = [time_checking])
    # content navigation
    rule12 = NavigationRule(action=anything, area=anything, status=anything, label = [content_navigation], area_count = 4)
    # irrelevant navigation
    rule13 = NavigationRule(action=anything, area=irrelevant_area, status=anything, label = [ir_content_navigation], area_count = 3)
    # irrelevant reading
    rule14 = DurationRule(action = anything, area = irrelevant_area, status = anything, label = [ir_reading], threshold = 3)
    rule15 = DurationRule(action = '', area = [''], status = status_off, label = [ir_reading], threshold = 3)
    # progress review
    rule16 = ProgressCheckRule(action = anything, area = [video_list], status = anything, label = [progress_check], signal_area = [reading_area])
    rules = [rule1, rule2, rule3, rule4, rule5, rule6, rule7, rule8, rule9, rule10, rule11, rule12, rule13, rule14, rule15, rule16]
    return rules

In [7]:
def label(df, filename):
    robotic_labels = []
    df_labelled = df.fillna('')
    rules = make_rules()
    for index, row in df_labelled.iterrows():
        label = []
        for rule in rules:
            label += rule.check(row)
        new_label = ""
        for l in label:
            if len(l) == 0:
                pass
            elif type(l) == str:
                new_label += l + ", "
            else:
                for i in range(len(l)):
                    if len(robotic_labels[-(i+1)]) == 0 or robotic_labels[-(i+1)][-2] == ',':
                        robotic_labels[-(i+1)] += l[i]
                    else:
                        robotic_labels[-(i+1)] += ', ' + l[i]
        if type(new_label) == str and len(new_label) > 0 and new_label[-2] == ',':
            new_label = new_label[:-2]
        robotic_labels.append(new_label)
    df_labelled['label'] = robotic_labels
    df_labelled.to_excel(filename)

In [8]:
for filename in os.listdir('data_video/eye tracking'):
    file_path = os.path.join('data_video/eye tracking', filename)
    name = filename.split('.')[0]
    if name != "20250725_subject62":
        continue
    log_name = 'log-' + name + '.xlsx'
    log_path = os.path.join('data_video/log', log_name)
    df = preprocess(file_path, log_path)
    subject_name = name.split('_')[-1]
    subject_number = subject_name.split('subject')[-1].zfill(2)
    label(df, 'p' + subject_number + '.xlsx')

C:\Users\jiang_c\AppData\Local\Temp\ipykernel_29616\3952018865.py:2: DtypeWarning: Columns (66,67) have mixed types. Specify dtype option on import or set low_memory=False.
  df_eye = pd.read_csv(filename, skiprows=5)


In [12]:
import pandas as pd
import numpy as np
from pathlib import Path

# Set root directory 
root_dir = Path(r"D:\research assistant\EEG\imotion data\data\eye tracking\raw")

# Find all CSVs in 'iMotion-csv' subfolders under any 'Subject XXX' directory
csv_files = list(root_dir.glob(r"*.csv"))

def process_file(raw_file):
    try:
        print(f"Processing: {raw_file}")
        formatted_file = raw_file.with_name(raw_file.stem + '_millisecond.csv')
        summarized_file = raw_file.with_name(raw_file.stem + '_1Hz_summarized.csv')

        # --- STEP 1: FORMAT RAW DATA ---
        raw = pd.read_csv(raw_file, header=None)
        rec_row = raw[raw.iloc[:, 0] == '#Recording time'].iloc[0]
        date_str = rec_row[1].split(':', 1)[1].strip()
        time_str = rec_row[2].split(':', 1)[1].strip()
        start_dt = pd.to_datetime(f"{date_str} {time_str}", format='%m/%d/%Y %H:%M:%S.%f %z').tz_localize(None)
        data_hdr_idx = raw.index[raw.iloc[:, 0] == '#DATA'][0] + 1
        orig_cols = raw.iloc[data_hdr_idx].tolist()
        df = raw.iloc[data_hdr_idx + 1:].reset_index(drop=True)
        df.columns = orig_cols

        drop_cols = [
            'SlideEvent', 'StimType', 'Duration', 'Reason', 'Name',
            'CollectionPhase', 'SourceStimuliName', 'SampleNumber',
            'Event Group', 'Event Label', 'Event Text', 'Event Index',
            'LSL Timestamp', 'Timestamp(1)'
        ]
        df = df.drop(columns=[c for c in drop_cols if c in df.columns])
        df['Timestamp'] = pd.to_numeric(df['Timestamp'], errors='coerce')
        df.insert(df.columns.get_loc('Timestamp') + 1, 'Datetime',
                  (start_dt + pd.to_timedelta(df['Timestamp'] / 1000.0, unit='s')).dt.strftime('%Y-%m-%d %H:%M:%S.%f').str[:-3])

        for col in ['Fixation Start', 'Fixation End', 'Saccade Start', 'Saccade End', 'ET_TimeSignal']:
            if col in df.columns:
                df[col] = (start_dt + pd.to_timedelta(pd.to_numeric(df[col], errors='coerce') / 1000.0, unit='s')).dt.strftime('%Y-%m-%d %H:%M:%S.%f').str[:-3]

        meta_tags = ['#Recording time', '#Device', '#Category', '#Description', '#Unit']

        def meta_row(tag):
            src = raw[raw.iloc[:, 0] == tag].iloc[0]
            row = [''] * len(df.columns)
            for j, name in enumerate(orig_cols):
                if name in drop_cols:
                    continue
                for idx in [i for i, c in enumerate(df.columns) if c == name]:
                    row[idx] = src.iloc[j]
            row[0] = tag
            return row

        header_block = pd.DataFrame([meta_row(t) for t in meta_tags], columns=df.columns)
        header_block.to_csv(formatted_file, index=False, header=False, encoding="utf-8-sig")
        df.to_csv(formatted_file, mode='a', index=False, header=True, encoding="utf-8-sig")

        # --- STEP 2: 1HZ SUMMARIZATION ---
        with open(formatted_file, encoding="utf-8-sig") as f:
            skiprows = 0
            meta_header_rows = []
            while True:
                line = f.readline()
                if not line or not line.startswith("#"):
                    break
                meta_header_rows.append(line.strip("\r\n").split(","))
                skiprows += 1

        df2 = pd.read_csv(formatted_file, skiprows=skiprows, encoding="utf-8-sig")
        drop_col_indices = [i for i in [0, 1, 3] if i < len(df2.columns)]
        df2 = df2.drop(df2.columns[drop_col_indices], axis=1)
        for col in ["SampleNumber1", "StimType1"]:
            if col in df2.columns:
                df2 = df2.drop(columns=col)
        df2 = df2.rename(columns={df2.columns[0]: "Datetime"})
        df2['Datetime'] = pd.to_datetime(df2['Datetime'], errors='coerce')
        df2 = df2.dropna(subset=['Datetime'])
        if df2.empty:
            print(f"Skipping {raw_file}: No valid 'Datetime' rows found.")
            return

        start_time = df2['Datetime'].iloc[0]
        end_time = df2['Datetime'].iloc[-1]
        resampled_times = pd.date_range(start=start_time.floor('s'), end=end_time.floor('s'), freq='1s')
        timeline = pd.DataFrame({'Resampled_Datetime': resampled_times})
        timeline['Timestamp'] = timeline['Resampled_Datetime'].dt.strftime('%m/%d/%Y %H:%M:%S')
        df2['Resampled_Datetime'] = df2['Datetime'].dt.floor('s')

        col_list = list(df2.columns)
        affectiva_cols = col_list[1:63]
        inputevent_cols = col_list[63:65]
        gazepoint_cols = col_list[65:78]
        emotiv_cols = col_list[78:84]
        imotions_cols = col_list[84:109]

        meta_names = [row[0] for row in meta_header_rows]
        meta_dict = {name: meta_header_rows[i][3:] if i < len(meta_header_rows) else [''] * (len(col_list) - 3) for i, name in enumerate(meta_names)}

        def filter_numeric_cols(df, cols):
            numeric_cols = []
            for c in cols:
                try:
                    pd.to_numeric(df[c].dropna(), errors='raise')
                    numeric_cols.append(c)
                except Exception:
                    pass
            return numeric_cols

        output_data = {'Timestamp': timeline['Timestamp']}
        header_rows = {name: [name] for name in meta_names}

        def agg_block(cols, block_name, stats=['min','mean','max']):
            numeric_cols = filter_numeric_cols(df2, cols)
            for c in cols:
                if c in numeric_cols:
                    group = df2.groupby('Resampled_Datetime')[c].agg(stats).reindex(resampled_times)
                    for stat in stats:
                        outcol = f"{c}_{stat}_{block_name}"
                        output_data[outcol] = group[stat].values
                        for name in meta_names:
                            idx = col_list.index(c)
                            header_rows[name].append(meta_dict[name][idx] if idx < len(meta_dict[name]) else "")
                else:
                    vals = df2.groupby('Resampled_Datetime')[c].last().reindex(resampled_times, method='ffill')
                    output_data[c] = vals.values
                    for name in meta_names:
                        idx = col_list.index(c)
                        header_rows[name].append(meta_dict[name][idx] if idx < len(meta_dict[name]) else "")

        # Affectiva
        agg_block(affectiva_cols, "affectiva")

        # Input Event (special aggregation)
        input_event_source_col, data_col = inputevent_cols
        def mouse_if_present(series):
            return "Mouse" if series.notnull().any() else np.nan
        def concat_nonempty(series):
            nonempty = [str(x) for x in series if pd.notnull(x) and str(x).strip()]
            return ".".join(nonempty) if nonempty else np.nan
        input_event_agg = df2.groupby('Resampled_Datetime').agg({
            input_event_source_col: mouse_if_present,
            data_col: concat_nonempty
        }).reindex(resampled_times)
        data_count = df2.groupby('Resampled_Datetime')[data_col].apply(lambda x: x.notnull().sum()).reindex(resampled_times)
        input_event_agg['Data_count'] = data_count
        for col in inputevent_cols:
            output_data[col] = input_event_agg[col].values
            for name in meta_names:
                idx = col_list.index(col)
                header_rows[name].append(meta_dict[name][idx] if idx < len(meta_dict[name]) else "")
        output_data['Data_count'] = input_event_agg['Data_count'].values
        for name in meta_names:
            if name == "#Description":
                header_rows[name].append("Count of non-empty Data per 1s")
            else:
                header_rows[name].append('')

        # Gazepoint
        agg_block(gazepoint_cols, "gazepoint")

        # Emotiv (last value)
        for c in emotiv_cols:
            vals = df2.groupby('Resampled_Datetime')[c].last().reindex(resampled_times, method='ffill')
            output_data[c] = vals.values
            for name in meta_names:
                idx = col_list.index(c)
                header_rows[name].append(meta_dict[name][idx] if idx < len(meta_dict[name]) else "")

        # iMotions
        agg_block(imotions_cols, "imotions")

        # --- Save summarized file with meta headers, starting from "#Device" ---
        meta_names_no_rec = [name for name in meta_names if name != "#Recording time"]
        with open(summarized_file, 'w', encoding="utf-8-sig") as f:
            for name in meta_names_no_rec:
                f.write(",".join(header_rows[name]) + "\n")
            f.write(",".join(output_data.keys()) + "\n")
        pd.DataFrame(output_data).to_csv(summarized_file, mode='a', index=False, header=False, encoding="utf-8-sig")
        print(f"Saved: {summarized_file}")

    except Exception as e:
        print(f"Error processing {raw_file}: {e}")

# Run for all files found
for file in csv_files:
    process_file(file)

Processing: D:\research assistant\EEG\imotion data\data\eye tracking\raw\20250603_subject38.csv


C:\Users\jiang_c\AppData\Local\Temp\ipykernel_24464\618147812.py:18: DtypeWarning: Columns (0,1,3,4,5,6,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124) have mixed types. Specify dtype option on import or set low_memory=False.
  raw = pd.read_csv(raw_file, header=None)


Saved: D:\research assistant\EEG\imotion data\data\eye tracking\raw\20250603_subject38_1Hz_summarized.csv
Processing: D:\research assistant\EEG\imotion data\data\eye tracking\raw\20250612_subject43.csv


C:\Users\jiang_c\AppData\Local\Temp\ipykernel_24464\618147812.py:18: DtypeWarning: Columns (0,1,3,4,5,6,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124) have mixed types. Specify dtype option on import or set low_memory=False.
  raw = pd.read_csv(raw_file, header=None)
C:\Users\jiang_c\AppData\Local\Temp\ipykernel_24464\618147812.py:71: DtypeWarning: Columns (66,67) have mixed types. Specify dtype option on import or set low_memory=False.
  df2 = pd.read_csv(formatted_file, skiprows=skiprows, encoding="utf-8-sig")


Saved: D:\research assistant\EEG\imotion data\data\eye tracking\raw\20250612_subject43_1Hz_summarized.csv
Processing: D:\research assistant\EEG\imotion data\data\eye tracking\raw\20250725_subject62.csv


...

Saved: D:\research assistant\EEG\imotion data\data\eye tracking\raw\20250825_subject93_1Hz_summarized.csv
Processing: D:\research assistant\EEG\imotion data\data\eye tracking\raw\20250826_subject93.csv


C:\Users\jiang_c\AppData\Local\Temp\ipykernel_24464\618147812.py:18: DtypeWarning: Columns (0,1,3,4,5,6,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124) have mixed types. Specify dtype option on import or set low_memory=False.
  raw = pd.read_csv(raw_file, header=None)
C:\Users\jiang_c\AppData\Local\Temp\ipykernel_24464\618147812.py:71: DtypeWarning: Columns (66,67) have mixed types. Specify dtype option on import or set low_memory=False.
  df2 = pd.read_csv(formatted_file, skiprows=skiprows, encoding="utf-8-sig")


Saved: D:\research assistant\EEG\imotion data\data\eye tracking\raw\20250826_subject93_1Hz_summarized.csv
Processing: D:\research assistant\EEG\imotion data\data\eye tracking\raw\20250913_subject102.csv


C:\Users\jiang_c\AppData\Local\Temp\ipykernel_24464\618147812.py:18: DtypeWarning: Columns (0,1,3,4,5,6,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124) have mixed types. Specify dtype option on import or set low_memory=False.
  raw = pd.read_csv(raw_file, header=None)


Saved: D:\research assistant\EEG\imotion data\data\eye tracking\raw\20250913_subject102_1Hz_summarized.csv
